In [1]:
import os
import sys

PROJECT_ROOT = os.getcwd()
sys.path.append(PROJECT_ROOT)



In [ ]:
#torch → core PyTorch library
#torch.nn (nn) → neural network layers (Conv2D, Linear, ReLU, etc.)
#torch.optim → optimizers like Adam
#torchvision → datasets and image transformations
#matplotlib → for visualizing images and results

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

In [ ]:
# This cell selects where the model will run:

# - GPU (CUDA) if available → much faster training
# - CPU otherwise

# The selected device is stored in the variable `device`
# and MUST exist before moving the model to it.

# Every tensor and model must be moved to the same device.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


In [ ]:
# This cell imports the CustomCNN class that we defined earlier.

# CustomCNN is our own convolutional neural network architecture
# designed specifically for Tiny ImageNet (200 classes).

In [ ]:
from src.models.custom_cnn import CustomCNN

model = CustomCNN(num_classes=200).to(device)
print(model)


In [ ]:
# This cell defines where the Tiny ImageNet dataset is located.
#
# PROJECT_ROOT → main project folder
# DATA_DIR     → points to "data/tiny-imagenet-200"
#
# TRAIN_DIR → folder containing training images
# VAL_DIR   → folder containing validation images
#
# os.path.join() safely builds paths that work on any system.
#
# The print statements check whether the train and val folders
# actually exist. If they print False, dataset loading will fail.

In [ ]:
DATA_DIR = os.path.join(PROJECT_ROOT, "data", "tiny-imagenet-200")

TRAIN_DIR = os.path.join(DATA_DIR, "train")
VAL_DIR   = os.path.join(DATA_DIR, "val")

print("Train exists:", os.path.exists(TRAIN_DIR))
print("Val exists:", os.path.exists(VAL_DIR))


In [ ]:
# This cell defines how images are processed before training.

# transforms.Resize((64, 64))
# → Resizes all images to 64x64 pixels (Tiny ImageNet standard).

# transforms.RandomHorizontalFlip()
# → Randomly flips images left-right during training.
# → This helps the model generalize better (data augmentation).

# transforms.ToTensor()
# → Converts images into PyTorch tensors
# → Scales pixel values from [0,255] to [0,1].

# Training and validation transforms are kept separate because
# validation data should NOT be augmented.

In [ ]:
train_transforms = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
])

val_transforms = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
])


In [ ]:
# This cell loads images from disk and prepares them for training.

# datasets.ImageFolder:
# → Automatically assigns labels based on folder names.
# → Each subfolder represents one class.

# DataLoader:
# → Loads data in small batches instead of all at once.
# → shuffle=True for training so the model does not memorize order.
# → shuffle=False for validation to keep results consistent.
# → batch_size=64 means 64 images are processed at a time.

# The print statements show how many images are available
# in the training and validation datasets.

In [ ]:
train_dataset = datasets.ImageFolder(TRAIN_DIR, transform=train_transforms)
val_dataset   = datasets.ImageFolder(VAL_DIR, transform=val_transforms)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=2)
val_loader   = DataLoader(val_dataset, batch_size=64, shuffle=False, num_workers=2)

print("Train samples:", len(train_dataset))
print("Val samples:", len(val_dataset))


Train samples: 100000
Val samples: 10000


In [ ]:
# This cell creates the model and prepares it for training.
#
# CustomCNN(num_classes=200)
# → Creates the CNN model for 200 Tiny ImageNet classes.
#
# .to(device)
# → Moves the model to GPU or CPU.
#
# nn.CrossEntropyLoss()
# → Loss function for multi-class classification.
#
# optim.Adam(...)
# → Adam optimizer updates model weights.
# → lr=0.001 is the learning rate (step size).
#


# print(model)
# → Displays the full model architecture


In [ ]:
model = CustomCNN(num_classes=200).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

print(model)


CustomCNN(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (5): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (8): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (10): ReLU()
    (11): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=16384, out_features=512, bias=True)
    (2): ReLU()
    (3): Dr

In [ ]:
#  TRAINING FUNCTION (ONE EPOCH)
# This function trains the model for ONE full pass over the training dataset.

# model.train()
# → Puts the model into training mode.
# → Enables layers like Dropout and BatchNorm to behave correctly.

# running_loss → tracks total loss for the epoch.
# correct      → counts correct predictions.
# total        → counts total images seen.

# For every batch:
# - Images and labels are moved to CPU/GPU.
# - optimizer.zero_grad() clears old gradients.
# - model(images) performs forward pass.
# - criterion computes how wrong predictions are.
# - loss.backward() computes gradients.
# - optimizer.step() updates model weights.

# torch.max(outputs, 1):
# → Picks the class with the highest score.

# Accuracy is calculated using correct / total.

# The function returns:
# - Average loss for the epoch
# - Training accuracy for the epoch


In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    return running_loss / len(loader), correct / total


In [ ]:
# This function evaluates the model on validation data.
#
# model.eval()
# → Puts the model into evaluation mode.
# → Disables Dropout and uses BatchNorm running averages.
#
# torch.no_grad()
# → Stops gradient calculation.
# → Saves memory and speeds up evaluation.
#
# No backpropagation or weight updates happen here.
#
# The loop:
# - Moves data to device.
# - Performs forward pass.
# - Computes loss.
# - Calculates accuracy.
#
# The function returns:
# - Average validation loss
# - Validation accuracy
#
# This is used to check how well the model generalize


In [ ]:
def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item()
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    return running_loss / len(loader), correct / total


In [ ]:
EPOCHS = 5

for epoch in range(EPOCHS):
    train_loss, train_acc = train_one_epoch(
        model, train_loader, criterion, optimizer, device
    )

    val_loss, val_acc = evaluate(
        model, val_loader, criterion, device
    )

    print(f"Epoch {epoch+1}/{EPOCHS}")
    print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}")
    print(f"Val   Loss: {val_loss:.4f}, Val   Acc: {val_acc:.4f}")
    print("-" * 40)
